# 💵 Currency Note Detector
### Upload any currency image → the trained model predicts what it is

**How to use this notebook:**
1. Run **Cell 1** — imports
2. Run **Cell 2** — loads the trained model saved by the training notebook
3. Run **Cell 3** — shows the class names learned during training
4. Run **Cell 4** — displays an upload button; pick any currency image and the model predicts it instantly

> **Prerequisite:** Run `Currency_Notes_Classification_Clean.ipynb` first and save the model as `currency_model.h5` (see the last cell of that notebook).

## Cell 1 — Imports

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import io
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

import tensorflow as tf
from tensorflow import keras

print(f"TensorFlow  : {tf.__version__}")
print(f"Keras       : {keras.__version__}")
print("✅ All imports successful")

## Cell 2 — Load the Trained Model

In [ ]:
# ──────────────────────────────────────────────────────────
# CONFIGURATION — edit these paths if your files are elsewhere
# ──────────────────────────────────────────────────────────
MODEL_PATH      = "currency_model.h5"     # saved model from training notebook
CLASS_JSON_PATH = "class_indices.json"    # optional: class-name map
DATASET_DIR     = "datasets"              # fallback: infer class names from folders
IMG_SIZE        = (224, 224)              # must match what the model was trained on
# ──────────────────────────────────────────────────────────

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model file '{MODEL_PATH}' not found.\n"
        "Please run the training notebook first and save the model:\n"
        "  model.save('currency_model.h5')"
    )

model = keras.models.load_model(MODEL_PATH)
print(f"✅ Model loaded from '{MODEL_PATH}'")
model.summary()

## Cell 3 — Discover Class Names

In [ ]:
# ── Priority 1: JSON file written by training notebook ──
if os.path.exists(CLASS_JSON_PATH):
    with open(CLASS_JSON_PATH) as f:
        idx_to_class = {int(v): k for k, v in json.load(f).items()}
    class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
    print(f"✅ Class names loaded from '{CLASS_JSON_PATH}'")

# ── Priority 2: Infer from dataset folder structure ──
elif os.path.isdir(DATASET_DIR):
    class_names = sorted([
        d for d in os.listdir(DATASET_DIR)
        if os.path.isdir(os.path.join(DATASET_DIR, d))
    ])
    print(f"✅ Class names inferred from '{DATASET_DIR}/' folder")

# ── Fallback: generic labels ──
else:
    num_classes = model.output_shape[-1]
    class_names = [f"currency_{i}" for i in range(num_classes)]
    print("⚠️  Could not find class names — using generic labels.")
    print("   Create 'class_indices.json' in the training notebook to fix this.")

print(f"\n🗂  {len(class_names)} classes detected:")
for i, name in enumerate(class_names):
    print(f"  [{i:>3}]  {name}")

## Cell 4 — Upload & Detect Currency Note

In [ ]:
# ── helper ──────────────────────────────────────────────────
def preprocess_image(pil_img: Image.Image) -> np.ndarray:
    """Resize → RGB → normalise → add batch dim."""
    img = pil_img.convert("RGB").resize(IMG_SIZE)
    arr = np.array(img, dtype=np.float32) / 255.0
    return np.expand_dims(arr, axis=0)   # shape (1, H, W, 3)


def predict_currency(pil_img: Image.Image):
    """Run inference and return (class_name, confidence, all_probs)."""
    tensor   = preprocess_image(pil_img)
    probs    = model.predict(tensor, verbose=0)[0]   # shape (num_classes,)
    idx      = int(np.argmax(probs))
    label    = class_names[idx] if idx < len(class_names) else f"class_{idx}"
    return label, float(probs[idx]), probs


def render_results(pil_img: Image.Image, label: str, confidence: float, probs: np.ndarray):
    """Display the uploaded image alongside a bar chart of top-5 probabilities."""
    top_n   = min(5, len(class_names))
    top_idx = np.argsort(probs)[::-1][:top_n]
    top_labels = [class_names[i] if i < len(class_names) else f"class_{i}" for i in top_idx]
    top_probs  = probs[top_idx]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor("#F8F9FA")

    # Left — uploaded image
    axes[0].imshow(pil_img)
    axes[0].axis("off")
    axes[0].set_title("Uploaded Image", fontsize=13, fontweight="bold", pad=10)

    # Right — horizontal bar chart
    colors = ["#2ECC71" if t == label else "#AED6F1" for t in top_labels]
    bars   = axes[1].barh(top_labels[::-1], top_probs[::-1] * 100, color=colors[::-1], edgecolor="white")
    axes[1].set_xlabel("Confidence (%)", fontsize=11)
    axes[1].set_xlim(0, 110)
    axes[1].set_title(f"Top-{top_n} Predictions", fontsize=13, fontweight="bold", pad=10)
    axes[1].spines[["top", "right"]].set_visible(False)

    for bar, prob in zip(bars[::-1], top_probs):
        axes[1].text(bar.get_width() + 1.5, bar.get_y() + bar.get_height() / 2,
                     f"{prob*100:.1f}%", va="center", fontsize=10)

    green_patch = mpatches.Patch(color="#2ECC71", label="Predicted class")
    blue_patch  = mpatches.Patch(color="#AED6F1", label="Other classes")
    axes[1].legend(handles=[green_patch, blue_patch], loc="lower right", fontsize=9)

    fig.suptitle(
        f"🏦  Predicted: {label}   |   Confidence: {confidence*100:.2f}%",
        fontsize=15, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    plt.show()


# ── UI ──────────────────────────────────────────────────────
upload_widget = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="📂 Upload Note",
    layout=widgets.Layout(width="200px")
)
detect_btn = widgets.Button(
    description="🔍 Detect Currency",
    button_style="success",
    layout=widgets.Layout(width="200px", height="36px")
)
status_lbl  = widgets.Label(value="Upload an image, then click Detect.")
output_area = widgets.Output()


def on_detect_clicked(_):
    with output_area:
        clear_output(wait=True)
        if not upload_widget.value:
            status_lbl.value = "⚠️  Please upload an image first."
            return

        # ipywidgets ≥ 8 stores uploads as a dict keyed by filename
        file_info = next(iter(upload_widget.value.values())) \
                    if isinstance(upload_widget.value, dict) \
                    else upload_widget.value[0]
        raw_bytes = file_info["content"] if isinstance(file_info, dict) else bytes(file_info["content"])

        try:
            pil_img = Image.open(io.BytesIO(raw_bytes))
        except Exception as e:
            status_lbl.value = f"❌ Could not open image: {e}"
            return

        status_lbl.value = "⏳ Running inference…"
        label, confidence, probs = predict_currency(pil_img)
        status_lbl.value = f"✅ Done — {label} ({confidence*100:.2f}%)"
        render_results(pil_img, label, confidence, probs)


detect_btn.on_click(on_detect_clicked)

display(
    HTML("<h3 style='color:#2C3E50'>🪙 Currency Note Detector</h3>"),
    widgets.HBox([upload_widget, detect_btn]),
    status_lbl,
    output_area
)

## Cell 5 — (Optional) Batch-test with a folder of images

In [ ]:
# Change TEST_FOLDER to any directory that contains images
TEST_FOLDER = "test_images"   # <── edit this

SUPPORTED = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not os.path.isdir(TEST_FOLDER):
    print(f"Folder '{TEST_FOLDER}' not found — skipping batch test.")
else:
    image_paths = [
        os.path.join(TEST_FOLDER, f)
        for f in sorted(os.listdir(TEST_FOLDER))
        if os.path.splitext(f)[1].lower() in SUPPORTED
    ]
    print(f"Found {len(image_paths)} images in '{TEST_FOLDER}'\n")

    results = []
    for path in image_paths:
        img   = Image.open(path)
        label, conf, _ = predict_currency(img)
        results.append({"file": os.path.basename(path), "prediction": label, "confidence": f"{conf*100:.2f}%"})
        print(f"  {os.path.basename(path):40s}  →  {label}  ({conf*100:.2f}%)")

    # Optional: save results to CSV
    import pandas as pd
    df = pd.DataFrame(results)
    df.to_csv("batch_results.csv", index=False)
    print("\n✅ Results saved to batch_results.csv")
    display(df)

---
## Appendix — Add this to the END of your Training Notebook

Paste the cell below into **`Currency_Notes_Classification_Clean.ipynb`** after you finish training so the detector notebook can load the model and class names.

```python
import json

# ── Save model ──
model.save('currency_model.h5')
print('✅ Model saved as currency_model.h5')

# ── Save class-index map (works for ImageDataGenerator / tf.data) ──
# If you used ImageDataGenerator:
#   class_indices = train_generator.class_indices
# If you used tf.keras.utils.image_dataset_from_directory:
#   class_indices = {name: i for i, name in enumerate(train_ds.class_names)}

with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=2)
print('✅ Class map saved as class_indices.json')
print('Classes:', list(class_indices.keys()))
```